# Top82

MBB beam topology optimization using the Helmholtz PDE sensitivity filter
from Andreassen et al. (2011). The target MATLAB call is
`top82(150,50,0.5,3,6,1)`.

In [ ]:
import jax
import jax.numpy as np

jax.config.update("jax_enable_x64", True)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax_fem import logger
from PIL import Image as PILImage

logger.setLevel("WARNING")

from topax.filter import HelmholtzFilter
from topax.optimizer import OC

In [ ]:
'''

2D MBB beam 

See https://link.springer.com/article/10.1007/s00158-010-0594-7

'''

import jax
from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper

from topax.problem import TopOptProblem


class Elasticity(TopOptProblem):

    def custom_init(self):
        self.fe = self.fes[0]
        self.fe.flex_inds = np.arange(len(self.fe.cells))

    def get_tensor_map(self):
        def stress(u_grad, xPhys):
            # material interpolation
            E = 1e-9 + xPhys**3 * (1.0 - 1e-9)
            nu = 0.3
            # Plane strain
            mu = E/(2.*(1.+nu))
            lmbda = E*nu/((1+nu)*(1-2*nu)) 
            # Plane strain -> Plane Stress
            lmbda = 2*mu*lmbda/(lmbda+2*mu) 
            epsilon = 0.5*(u_grad + u_grad.T)
            sigma = lmbda * np.trace(epsilon) * np.eye(self.dim) + 2*mu*epsilon
            return sigma
        return stress

    def set_params(self, params):
        # Override base class method.
        full_params = np.ones((self.fe.num_cells, params.shape[1]))
        full_params = full_params.at[self.fe.flex_inds].set(params)
        thetas = np.repeat(full_params[:, None, :], self.fe.num_quads, axis=1)
        self.full_params = full_params
        self.internal_vars = [thetas]

    def compute_compliance(self, sol):
        return np.sum(self.point_force * sol[self.load_node])


def prep_fem(Nx, Ny, Lx, Ly):

    # Mesh
    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(Nx, Ny, domain_x=Lx, domain_y=Ly)
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type], ele_type)

    # BCs
    def left(point):
        return np.isclose(point[0], 0., atol=1e-5)

    def right_bottom_corner(point):
        return np.logical_and(np.isclose(point[0], Lx, atol=1e-5), 
                              np.isclose(point[1], 0., atol=1e-5))

    def left_middle(point):
        return np.logical_and(np.isclose(point[0], 0., atol=1e-5), 
                              np.isclose(point[1], Ly, atol=1e-5))

    def dirichlet_val(point):
        return 0.

    dirichlet_bc_info = [[left, right_bottom_corner], 
                         [0, 1], [dirichlet_val]*2]
    # Problem
    problem = Elasticity(mesh, vec=2, dim=2, ele_type=ele_type, 
                         dirichlet_bc_info=dirichlet_bc_info)

    point_force = np.array([0., -1.])
    problem.point_force = point_force
    problem.load_node = problem.add_point_load(left_middle, point_force)

    # Differentiable wrapper
    solver_options = {'petsc_solver':{'ksp_type': 'preonly', 'pc_type': 'lu'}}
    fwd_pred = ad_wrapper(problem, 
                          solver_options=solver_options,
                          adjoint_solver_options=solver_options)

    return fwd_pred, problem

In [ ]:
# SETUP
vf = 0.5
rmin = 6.0

# forward model
Nx, Ny = 150, 50
Lx, Ly = Nx, Ny
fwd_pred, problem = prep_fem(Nx, Ny, Lx, Ly)

# objective function
def J_total(xPhys):
    sol_list = fwd_pred(xPhys)
    compliance = problem.compute_compliance(sol_list[0])
    return compliance

# constraint function
def volume_constraints(xPhys):
    g = np.sum(xPhys) - vf * xPhys.size
    return g

# Helmholtz PDE sensitivity filter
pde_filter = HelmholtzFilter(problem, rmin=rmin)

def sensitivity_filter(dc, xPhys):
    filtered = pde_filter(dc * xPhys)
    return filtered / np.maximum(1e-3, xPhys)

optimizer = OC(move=0.2, damping=0.5)

# initial design
x0 = vf * np.ones((Nx * Ny, 1))

In [ ]:
# OPTIMIZATION LOOP
loop = 0
change = 1
xnew = x0
frames = []
while change > 0.01:
    loop += 1
    # evaluate
    J, dJ = jax.value_and_grad(J_total)(xnew)
    c, dc = jax.value_and_grad(volume_constraints)(xnew)
    # filter
    xold = xnew.copy()
    dJ = sensitivity_filter(dJ, xold)
    # update
    xnew = optimizer.update(xold, dJ, dc, np.mean, vf)
    xPhys = xnew
    vol = np.mean(xPhys)
    change = np.max(np.abs(xnew - xold))
    print(f' It.:{loop:5d}, Obj.:{J:11.4f}, Vol.:{vol:7.3f}, ch.:{change:7.3f}')
    field = onp.flip(xPhys.reshape(Ny, Nx, order='F'), axis=0)
    frames.append(onp.asarray(field))

In [ ]:
# SAVE OPTIMIZATION HISTORY
output_path = Path("docs/imgs/example_top82.gif")
output_path.parent.mkdir(parents=True, exist_ok=True)

gif_frames = []
for field in frames:
    rgba = plt.get_cmap("gray_r")(onp.clip(field, 0.0, 1.0), bytes=True)
    image = PILImage.fromarray(rgba)
    image = image.resize((Nx * 8, Ny * 8), PILImage.Resampling.NEAREST)
    gif_frames.append(image)

gif_frames[0].save(
    output_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
)
display(DisplayImage(filename=str(output_path)))